# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, inspecting, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema located at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

All dataset entities (record sets, fields, columns) are referenced **by their `@id`** fields throughout the notebook, ensuring reproducible and precise data access.

In [ ]:
# Make sure mlcroissant and pandas are installed
%pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# The Croissant schema URL for this dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), their fields, and corresponding `@id` values.

**Note**: All data entities are referenced via their `@id`s as required by best Croissant practices.

In [ ]:
# List all available record sets and their IDs
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")

if not record_sets:
    print("No record sets found in Croissant schema.\n")
else:
    for rs in record_sets:
        print(f"- Record set name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields and their @id:")
        for field in rs.fields:
            print(f"    - {field.name} (id: {field.id}, datatype: {getattr(field, 'data_type', 'n/a')})")
        print()

## 3. Data Extraction
Load the data from each record set into a Pandas DataFrame for analysis.

Replace `record_set_id` and field IDs as needed from the output above. **Field and record set references should use their `@id` fields.**

In [ ]:
# Extract dataframes from available record sets using their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for RecordSet @id='{record_set_id}':")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Rows: {len(df)}")
    print()

# Example: Show head of first available record set
if record_set_ids:
    preview_id = record_set_ids[0]
    display_columns = dataframes[preview_id].columns.tolist()
    print(f"Preview of first records in RecordSet @id: {preview_id}")
    display(dataframes[preview_id].head())
else:
    print("No data available in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing and analysis: filtering, normalization, grouping, and other transformations. All columns used are referenced by their **`@id`** field.

We will:
- Select a numeric field (by `@id`) for filtering and normalization.
- Filter for values above a threshold.
- Normalize the filtered column.
- Optionally, group by a categorical field.

In [ ]:
# Choose target record set and numeric field by @id
numeric_field_id = None
group_field_id = None
target_record_set_id = None

# Let's inspect the available DataFrames and infer likely numeric fields by dtype
for record_set_id, df in dataframes.items():
    # Try to locate a numeric column
    num_cols = df.select_dtypes(include=['number', 'float', 'int']).columns
    if len(num_cols) > 0:
        numeric_field_id = num_cols[0]
        target_record_set_id = record_set_id
        break

if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Analyzing record set '@id': {target_record_set_id}")
    print(f"Numeric field for EDA: '@id' = {numeric_field_id}")

    # Find a candidate group field (non-numeric, non-date)
    group_candidates = [c for c in dataframes[target_record_set_id].columns if c != numeric_field_id and not pd.api.types.is_numeric_dtype(dataframes[target_record_set_id][c])] 
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by field '@id': {group_field_id}")

    # Filtering
    threshold = dataframes[target_record_set_id][numeric_field_id].median()  # Use median as default

    filtered_df = dataframes[target_record_set_id][dataframes[target_record_set_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (standard score):")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped means of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions and field relationships. The above EDA code picks the most suitable numeric field and (optionally) categorical field automatically, both by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and target_record_set_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[target_record_set_id][numeric_field_id].dropna(), bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field is available, show boxplot
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you learned how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all major entities by their unique `@id`. The workflow included dynamic data loading, column selection, filtering, normalization, grouping, and visualization -- all driven by the semantics of the Croissant schema.

**Key steps covered:**
- Loading dataset metadata and records
- Exploring available record sets, fields, and columns by `@id`
- Extracting and previewing DataFrames for each record set
- Performing EDA on numeric fields, with threshold filtering, normalization, and grouping
- Visualizing result distributions and relationships

You are encouraged to further explore each record set's schema and fields for advanced analytics!
